In [2]:
import os
import re
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import requests
import re
import unicodedata
from urllib.parse import urlparse

def title_to_filename(title):

    # Bỏ dấu tiếng Việt
    title = unicodedata.normalize("NFD", title)
  #  title = "".join(c for c in title if unicodedata.category(c) != "Mn")
    title = title.replace("đ", "d").replace("Đ", "D")

    # Tạo slug
    slug = re.sub(r"[^a-zA-Z0-9]+", "-", title.lower()).strip("-")

    return f"{slug}.md"


def crawl_to_markdown_with_images(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    # Tạo thư mục lưu trữ ảnh nếu chưa có
    image_dir = "images"
    if not os.path.exists(image_dir):
        os.makedirs(image_dir)

    try:
        print(f"Đang tải dữ liệu từ: {url}...")
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        response.encoding = "utf-8"
    except Exception as e:
        print(f"Lỗi khi tải trang: {e}")
        return

    soup = BeautifulSoup(response.text, "html.parser")
    markdown_content = []

    # 1. Lấy tiêu đề chính
    # Sử dụng select_one để lấy phần tử theo CSS selector
    title_tag = soup.select_one("body > div:nth-child(7) > center > table > tbody > tr > td:nth-child(2) > b:nth-child(1)")

    if title_tag:
        # Lấy text và làm sạch các khoảng trắng thừa
        title = title_tag.get_text(strip=True)
        markdown_content.append(f"# {title}\n")
        output_filename = title_to_filename(title)
    else:
        print("Không tìm thấy tiêu đề với cấu trúc HTML hiện tại.")    
        slug = urlparse(url).path.strip("/").split("/")[-1]
        output_filename =  f"{slug}.md"
        

    # 2. Tìm vùng nội dung chính
    content_area = soup.find("div", class_="td-post-content") or soup.find(
        "article"
    )
    source_area = content_area if content_area else soup.body

    if not source_area:
        print("Không tìm thấy vùng nội dung chính.")
        return

    # Quét qua các phần tử bao gồm cả thẻ div (thường chứa ảnh kèm chú thích)
    for element in source_area.find_all(
        ["h2", "h3", "p", "ul", "ol", "blockquote", "img", "figure"]
    ):

        # Xử lý nếu phần tử là thẻ IMG độc lập hoặc nằm trong thẻ khác
        if element.name == "img":
            img_url = element.get("src") or element.get("data-src")
            if img_url:
                # Chuyển đổi đường dẫn tương đối thành tuyệt đối
                full_img_url = urljoin(url, img_url)
                alt_text = element.get("alt", "Hinh_Anh").strip() or "Hình ảnh"

                # Đặt tên file ảnh dựa trên url để tránh trùng lặp
                img_name = os.path.basename(full_img_url.split("?")[0])
                if not img_name or not img_name.endswith(
                    (".jpg", ".jpeg", ".png", ".webp")
                ):
                    img_name = f"img_{len(os.listdir(image_dir)) + 1}.jpg"

                local_img_path = os.path.join(image_dir, img_name)

                # Tải ảnh về máy
                try:
                    img_data = requests.get(
                        full_img_url, headers=headers
                    ).content
                    with open(local_img_path, "wb") as img_file:
                        img_file.write(img_data)
                    # Chèn cú pháp hình ảnh vào Markdown (sử dụng dấu gạch chéo xuôi cho đường dẫn)
                    markdown_content.append(
                        f"\n![{alt_text}](images/{img_name})\n"
                    )
                except Exception as img_err:
                    print(f"Không thể tải ảnh {full_img_url}: {img_err}")
            continue

        # Bỏ qua các thẻ chứa ảnh đã được xử lý (tránh trùng văn bản bên trong figure)
        if element.name == "figure":
            continue

        text = element.get_text().strip()
        if not text:
            continue

        # Định dạng văn bản sang Markdown
        if element.name == "h2":
            markdown_content.append(f"\n## {text}\n")
        elif element.name == "h3":
            markdown_content.append(f"\n### {text}\n")
        elif element.name == "p":
            b_tag = element.find("b")
            
            # Kiểm tra xem thẻ p này có phải là một "tiêu đề giả h2" không
            # Điều kiện: có thẻ <b>, nội dung của p đúng bằng nội dung của <b>, và viết IN HOA
            if b_tag and text == b_tag.get_text(strip=True) and text.isupper():
                markdown_content.append(f"\n## {text}\n\n")
            else:
                # Nếu không, xử lý nó như một đoạn văn bản bình thường
                markdown_content.append(f"{text}\n\n")
                
        elif element.name in ["ul", "ol"]:
            for li in element.find_all("li"):
                li_text = li.get_text(strip=True) # Nên thêm strip=True để làm sạch text
                if li_text:
                    markdown_content.append(f"* {li_text}\n")
            markdown_content.append("\n")
        elif element.name == "blockquote":
            markdown_content.append(f"> {text}\n\n")
            
        # if element.name == "h2":
        #     markdown_content.append(f"\n## {text}\n")
        # elif element.name == "h3":
        #     markdown_content.append(f"\n### {text}\n")
        # elif element.name == "p":
        #     markdown_content.append(f"{text}\n")
        # elif element.name in ["ul", "ol"]:
        #     for li in element.find_all("li"):
        #         li_text = li.get_text().strip()
        #         if li_text:
        #             markdown_content.append(f"* {li_text}\n")
        #     markdown_content.append("\n")
        # elif element.name == "blockquote":
        #     markdown_content.append(f"> {text}\n\n")

    # 3. Ghi file Markdown
    try:
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write("".join(markdown_content))
        print(f"\nThành công!")
        print(f"- File bài viết: {os.path.abspath(output_filename)}")
        print(f"- Thư mục hình ảnh: {os.path.abspath(image_dir)}")
    except IOError as e:
        print(f"Lỗi khi ghi file: {e}")


# if __name__ == "__main__":
#     target_url = "https://vnras.com/cay-ngu-vi-tu-nhung-cay-thuoc-va-vi-thuoc-viet-nam-do-tat-loi/"
#     crawl_to_markdown_with_images(target_url)

from pathlib import Path

url_file = "tat_ca_url_tac_gia.txt"
done_file = "done_urls.txt"

done_urls = set()
if Path(done_file).exists():
    with open(done_file, "r", encoding="utf-8") as f:
        done_urls = {line.strip() for line in f if line.strip()}

with open(url_file, "r", encoding="utf-8") as f:
    urls = [line.strip() for line in f if line.strip()]

for i, target_url in enumerate(urls, start=1):
    if target_url in done_urls:
        continue

    try:
        print(f"[{i}/{len(urls)}] Crawling: {target_url}")
        crawl_to_markdown_with_images(target_url)

        with open(done_file, "a", encoding="utf-8") as f:
            f.write(target_url + "\n")

    except Exception as e:
        print(f"ERROR: {target_url}")
        print(e)    

[7/9] Crawling: https://www.budsas.org/uni/u-ducphatlichsu/dpls-06a.htm
Đang tải dữ liệu từ: https://www.budsas.org/uni/u-ducphatlichsu/dpls-06a.htm...
Không tìm thấy tiêu đề với cấu trúc HTML hiện tại.

Thành công!
- File bài viết: /Users/ng/projects/n5/.scripts/ducphatlichsu/dpls-06a.htm.md
- Thư mục hình ảnh: /Users/ng/projects/n5/.scripts/ducphatlichsu/images
[8/9] Crawling: https://www.budsas.org/uni/u-ducphatlichsu/dpls-07.htm
Đang tải dữ liệu từ: https://www.budsas.org/uni/u-ducphatlichsu/dpls-07.htm...
Không tìm thấy tiêu đề với cấu trúc HTML hiện tại.

Thành công!
- File bài viết: /Users/ng/projects/n5/.scripts/ducphatlichsu/dpls-07.htm.md
- Thư mục hình ảnh: /Users/ng/projects/n5/.scripts/ducphatlichsu/images
[9/9] Crawling: https://www.budsas.org/uni/u-ducphatlichsu/dpls-08.htm
Đang tải dữ liệu từ: https://www.budsas.org/uni/u-ducphatlichsu/dpls-08.htm...
Không tìm thấy tiêu đề với cấu trúc HTML hiện tại.

Thành công!
- File bài viết: /Users/ng/projects/n5/.scripts/ducphatli